# Task 2: Feature Engineering & Preprocessing Pipeline

## Objective
Build a reusable, leak-free scikit-learn preprocessing pipeline with missing-value imputation, numerical scaling, categorical one-hot encoding, correlation analysis, and feature-importance analysis.

**Dataset:** Titanic passenger dataset loaded from OpenML. The task page provides examples of suitable tabular datasets but does not provide an official dataset file, so this notebook uses a public mixed-type tabular dataset as the working example.

**Leakage control:** train/test splitting happens before fitting any preprocessing transformation. All learned transformations are inside the scikit-learn pipeline.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

RANDOM_STATE = 42

## 1. Load the dataset

In [ ]:
titanic = fetch_openml('titanic', version=1, as_frame=True)
df = titanic.frame.copy()

print('Shape:', df.shape)
display(df.head())
display(df.dtypes)

## 2. Basic quality checks

Check missingness, duplicates, target balance, and cardinality before preprocessing.

In [ ]:
print('Missing values:')
display(df.isna().sum().sort_values(ascending=False))

print('Duplicate rows:', df.duplicated().sum())
print('\nTarget balance:')
display(df['survived'].value_counts(dropna=False))


## 3. Select features and target

The target is `survived`. Identifier-like and redundant fields are excluded from modelling. The remaining features include both numerical and categorical variables and contain missing values.

In [ ]:
target = 'survived'
drop_cols = ['survived', 'boat', 'body', 'home.dest', 'name', 'ticket']
model_df = df.drop(columns=[c for c in drop_cols if c in df.columns]).copy()

X = model_df.drop(columns=[target])
y = model_df[target].astype(int)

numeric_features = X.select_dtypes(include=['number']).columns.tolist()
categorical_features = X.select_dtypes(exclude=['number']).columns.tolist()

print('Numeric features:', numeric_features)
print('Categorical features:', categorical_features)

## 4. Train/test split before transformations

Splitting first prevents information from the test set from influencing imputation, scaling, or encoding.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 5. Leak-free preprocessing pipeline

Numerical columns use median imputation followed by standard scaling. Categorical columns use most-frequent imputation followed by one-hot encoding. All preprocessing is fitted only on the training data through `ColumnTransformer`.

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

rf_pipeline = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight='balanced'
    ))
])

rf_pipeline.fit(X_train, y_train)

## 6. Evaluate the pipeline

In [ ]:
pred = rf_pipeline.predict(X_test)

print('Accuracy:', accuracy_score(y_test, pred))
print('Precision:', precision_score(y_test, pred, zero_division=0))
print('Recall:', recall_score(y_test, pred, zero_division=0))
print('F1:', f1_score(y_test, pred, zero_division=0))
print('\nConfusion matrix:\n', confusion_matrix(y_test, pred))

## 7. Correlation analysis

Correlation is calculated on numerical variables only. This is exploratory analysis and is not used to fit the model.

In [ ]:
corr = X_train[numeric_features].corr()
display(corr)

plt.figure(figsize=(8, 6))
plt.imshow(corr, interpolation='nearest')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.colorbar(label='Correlation')
plt.title('Numerical Feature Correlation')
plt.tight_layout()
plt.show()

## 8. Tree-based feature importance

After fitting the pipeline, retrieve the transformed feature names and Random Forest importances.

In [ ]:
feature_names = rf_pipeline.named_steps['preprocess'].get_feature_names_out()
importances = rf_pipeline.named_steps['model'].feature_importances_

importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': importances
}).sort_values('importance', ascending=False)

display(importance_df.head(20))

top = importance_df.head(15).sort_values('importance')
plt.figure(figsize=(9, 6))
plt.barh(top['feature'], top['importance'])
plt.xlabel('Random Forest importance')
plt.title('Top Transformed Feature Importances')
plt.tight_layout()
plt.show()

## 9. Mutual-information feature ranking

For an additional ranking, mutual information is computed after preprocessing. Because one-hot encoded columns are binary and numeric columns are continuous, the `discrete_features` mask is constructed from the transformed feature names.

In [ ]:
X_train_transformed = rf_pipeline.named_steps['preprocess'].transform(X_train)
discrete_mask = np.array([name.startswith('cat__') for name in feature_names])

mi = mutual_info_classif(
    X_train_transformed,
    y_train,
    discrete_features=discrete_mask,
    random_state=RANDOM_STATE
)

mi_df = pd.DataFrame({
    'feature': feature_names,
    'mutual_information': mi
}).sort_values('mutual_information', ascending=False)

display(mi_df.head(20))

## 10. Leakage and reproducibility checklist

- Train/test split occurs before fitting preprocessing.
- Imputation parameters are learned only from training data.
- Scaling parameters are learned only from training data.
- One-hot encoding is inside the pipeline and handles unseen categories.
- The model and preprocessing are stored in one reusable pipeline.
- Identifier-like and post-outcome fields are excluded from modelling.
- A fixed random seed is used for reproducibility.

## Conclusion
The pipeline demonstrates reusable preprocessing for mixed-type tabular data while reducing leakage risk. Feature importance and mutual-information rankings are included for interpretation. Model metrics should be interpreted together with dataset size, class balance, and the limitations of the selected public dataset.